# Laboratório 3 — Alinhamento, Homografia 2D e Mosaico

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2  
**Professor:** Celso Setsuo Kurashima

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** 17/06/2026  
**Data de publicação do relatório:** 29/06/2026

## Sumário

1. [Introdução](#1-introdução)
2. [Fundamentação teórica](#2-fundamentação-teórica)
   - 2.1 [Alinhamento de imagens e correspondência de feições](#21-alinhamento-de-imagens-e-correspondência-de-feições)
   - 2.2 [Homografia 2D](#22-homografia-2d)
   - 2.3 [Mínimos Quadrados Tradicionais vs. RANSAC](#23-mínimos-quadrados-tradicionais-vs-ransac)
   - 2.4 [Image Stitching (mosaico de imagens)](#24-image-stitching-mosaico-de-imagens)
3. [Ambiente e instruções de reprodução](#3-ambiente-e-instruções-de-reprodução)
4. [Procedimentos experimentais (Parte 2)](#4-procedimentos-experimentais-parte-2)
   - 4.1 [Captura das fotos pelas webcams](#41-captura-das-fotos-pelas-webcams)
   - 4.2 [(A) Alinhamento, homografia e mosaico em imagens gravadas](#42-a-alinhamento-homografia-e-mosaico-em-imagens-gravadas)
   - 4.3 [(B) Pipeline de homografia e mosaico em duas webcams ao vivo](#43-b-pipeline-de-homografia-e-mosaico-em-duas-webcams-ao-vivo)
5. [Análise e discussão](#5-análise-e-discussão)
   - 5.1 [Análise estatística das características (outliers)](#51-análise-estatística-das-características-outliers)
   - 5.2 [Álgebra linear e homografia](#52-álgebra-linear-e-homografia)
   - 5.3 [Análise geométrica e robótica](#53-análise-geométrica-e-robótica)
   - 5.4 [Qualidade do processamento de imagem (artefatos da costura)](#54-qualidade-do-processamento-de-imagem-artefatos-da-costura)
   - 5.5 [Aplicações e uso no trabalho final](#55-aplicações-e-uso-no-trabalho-final)
6. [Conclusões](#6-conclusões)
7. [Declaração de uso de Inteligência Artificial Generativa](#7-declaração-de-uso-de-inteligência-artificial-generativa)
8. [Referências](#8-referências)

## 1. Introdução

Nos laboratórios anteriores a equipe estudou a captura de imagens/vídeos (Lab 1) e a extração e
correspondência de **características** com SIFT (Lab 2). Neste Laboratório 3 damos o passo seguinte:
usar as correspondências entre duas imagens para **alinhá-las geometricamente** por meio de uma
**homografia 2D** e, a partir desse alinhamento, **costurar** as duas imagens em um único **mosaico**
(*Image Stitching*).

O laboratório tem um forte componente de **robustez matemática**. Em um cenário real — por exemplo, um
robô móvel ou um drone equipado com câmera — as correspondências obtidas contêm **ruído** e **falsos
positivos** (*outliers*). Por isso, comparamos dois métodos de estimativa da homografia: o de
**Mínimos Quadrados Tradicionais (MQT)**, que é sensível a outliers, e o **RANSAC**, que filtra as
correspondências de forma robusta. O relatório descreve a fundamentação teórica (Parte 1), os dois
programas implementados pela equipe (Parte 2 — captura/alinhamento em imagens gravadas e versão ao vivo
com duas webcams) e responde às perguntas de análise propostas no roteiro, incluindo a análise
estatística de outliers, a comparação das matrizes de homografia, a discussão geométrica/robótica e os
artefatos visuais da costura.

## 2. Fundamentação teórica

### 2.1 Alinhamento de imagens e correspondência de feições

**Alinhar** duas imagens significa encontrar a transformação geométrica que leva os pontos de uma
imagem às posições correspondentes na outra. Quando as duas imagens observam aproximadamente a **mesma
cena planar** (ou são tomadas a partir de um mesmo centro óptico, apenas com rotação da câmera), essa
transformação é uma **homografia**.

O ponto de partida é o mesmo do Lab 2: detectar **keypoints** e **descritores** com o **SIFT**
(`cv2.SIFT_create`), casar os descritores de uma imagem com os da outra e filtrar correspondências
ambíguas com o **teste de razão de Lowe**. Neste laboratório a equipe usou o **BFMatcher**
(*Brute-Force*) com `knnMatch(k=2)` e razão de Lowe de **0,75**: um match só é aceito se
`m.distance < 0.75 * n.distance`. O conjunto resultante (`bons_matches`) ainda pode conter outliers
geométricos — daí a necessidade do RANSAC na etapa seguinte.

### 2.2 Homografia 2D

Uma **homografia** é uma transformação projetiva entre dois planos, representada por uma matriz
$3\times3$ que opera sobre coordenadas **homogêneas**:

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} \sim H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}, \qquad H = \begin{bmatrix} h_{00} & h_{01} & h_{02} \\ h_{10} & h_{11} & h_{12} \\ h_{20} & h_{21} & h_{22} \end{bmatrix}$$

O símbolo $\sim$ indica igualdade **a menos de um fator de escala**: $H$ e $\lambda H$ representam a
mesma transformação. Por isso, embora a matriz tenha 9 elementos, ela possui apenas **8 graus de
liberdade** — costuma-se fixar a escala normalizando $h_{22} = 1$. Os elementos têm interpretação
geométrica: o bloco $2\times2$ superior esquerdo carrega rotação, escala e cisalhamento; $h_{02}$ e
$h_{12}$ são as **translações** horizontal e vertical; e $h_{20}, h_{21}$ codificam o efeito de
**perspectiva**. No OpenCV a homografia é estimada por `cv2.findHomography` e aplicada com
`cv2.warpPerspective`.

Cada par de pontos correspondentes $(x,y) \leftrightarrow (x',y')$ fornece **duas** equações lineares
(uma para $x'$, outra para $y'$). Como há 8 incógnitas, são necessários **no mínimo 4 pares de pontos**
(não colineares) para determinar $H$ — assunto retomado na análise.

### 2.3 Mínimos Quadrados Tradicionais vs. RANSAC

Tendo mais correspondências do que o mínimo, o sistema fica **sobredeterminado** e é resolvido por
otimização. Há duas filosofias:

- **Mínimos Quadrados Tradicionais (MQT).** Minimiza a soma dos erros quadráticos de **todas** as
  correspondências de uma só vez (no OpenCV, `cv2.findHomography(src, dst, method=0)`). É ótimo quando
  o ruído é gaussiano e **não há outliers**. O problema: como o erro é elevado ao quadrado, **poucos**
  outliers com erro grande dominam a soma e **puxam** a solução para longe do valor correto. É o
  método clássico, frágil em cenários reais e ruidosos (sensores robóticos).
- **RANSAC** (*RANdom SAmple Consensus*). Em vez de usar todos os pontos, sorteia repetidamente
  subconjuntos **mínimos** (4 pares), estima um $H$ candidato e conta quantas correspondências ficam
  dentro de um limiar de reprojeção (aqui, `reproj_threshold = 5.0` px) — os **inliers**. Após várias
  iterações, mantém o modelo com maior consenso e reestima $H$ usando só os inliers. Devolve também uma
  **máscara** (`mask_ransac`) marcando inliers (1) e outliers (0). É **robusto**: outliers simplesmente
  não entram no ajuste final.

A comparação direta entre `H_ls` (MQT) e `H_ransac` é justamente o coração analítico deste laboratório:
ela torna visível o quanto os outliers distorcem o método clássico.

### 2.4 Image Stitching (mosaico de imagens)

Com a homografia estimada, o **mosaico** é construído **projetando** uma imagem no plano da outra com
`cv2.warpPerspective` e sobrepondo a segunda imagem na região comum. No pipeline da equipe, a
`imagem1` é deformada (*warped*) para uma tela larga (`width = img1.w + img2.w`) e a `imagem2` é colada
por cima na sua posição original, formando o panorama.

O pipeline completo de *stitching* descrito na literatura (Szeliski; tutoriais LearnOpenCV/PyImageSearch)
envolve: (1) detecção e descrição de feições; (2) *matching* + filtragem; (3) estimativa robusta da
homografia (RANSAC); (4) *warping*/projeção; e (5) **blending** (costura fina) para suavizar a emenda.
Esta última etapa — tratada na análise — é o que elimina linhas de corte, saltos de iluminação e
*ghosting* na junção.

## 3. Ambiente e instruções de reprodução

Os experimentos foram executados em **Linux**, com **Python 3** e **OpenCV 4.x** (o SIFT já integra o
pacote principal). Para reproduzir:

```bash
# 1) Ambiente virtual
python3 -m venv venv
source venv/bin/activate

# 2) Dependências
pip install opencv-python numpy matplotlib

# 3) Capturar as fotos pelas webcams (item A)
python3 lab3_camera_foto.py     # salva imagem1.png / imagem2.png (tecla 'x' grava, 'q' sai)

# 4) Alinhamento + homografia (MQT e RANSAC) + mosaico em imagens gravadas
python3 lab3_.py

# 5) Versão ao vivo com duas webcams (item B)
python3 lab3_codigo_adaptado.py
```

> **Observação:** a captura e as janelas (`cv2.imshow`) exigem **interface gráfica**; o item (A) usa as
> duas câmeras para tirar as duas fotos e o item (B) lê **duas webcams simultâneas** (índices `0` e
> `1`). Os arquivos de resultado citados abaixo estão na mesma pasta deste notebook e são referenciados
> por caminhos relativos.

## 4. Procedimentos experimentais (Parte 2)

### 4.1 Captura das fotos pelas webcams

**Objetivo:** capturar, com a webcam, as duas fotos de entrada do experimento (`imagem1.png` e
`imagem2.png`), garantindo **área de sobreposição de pelo menos 50%** entre elas — a segunda foto é
tirada **rotacionando ligeiramente a câmera para o lado**, de modo que ambas observem a mesma cena
planar a partir de pontos de vista próximos. O programa exibe o vídeo ao vivo e grava o quadro atual ao
pressionar **`x`** (encerra com **`q`**).

In [ ]:
import numpy as np
import cv2 as cv

cap = cv.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

while True:
    # Captura quadro a quadro
    ret, frame = cap.read()
    # se o quadro foi lido corretamente, ret e True
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    # Exibe o quadro atual
    cv.imshow('frame', frame)

    if cv.waitKey(1) == ord('q'):
        break

    if cv.waitKey(1) == ord('x'):
        cv.imwrite('imagem2.png', frame)

# Libera a captura
cap.release()
cv.destroyAllWindows()

> **Nota da equipe.** O mesmo programa foi usado para gravar as duas entradas (alterando o nome do
> arquivo de saída para `imagem1.png` e `imagem2.png`). Conforme o roteiro, foram montados experimentos
> com uma **pessoa** e com um **objeto texturizado** (um livro/embalagem segurado na mão), pois objetos
> com textura rica geram muito mais keypoints estáveis do que uma pessoa ou uma parede lisa.

### 4.2 (A) Alinhamento, homografia e mosaico em imagens gravadas

**Objetivo:** sobre as duas fotos gravadas, executar o pipeline completo — SIFT → *matching* (BF + razão
de Lowe) → estimativa da homografia por **MQT** e por **RANSAC** → visualização das correspondências
inliers → **mosaico** comparando os dois métodos. O programa também imprime no console as matrizes
`H_ls` e `H_ransac` e a contagem de inliers/outliers, dados usados na seção de análise.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# =====================================================================
# CONFIGURACAO E CARREGAMENTO DAS IMAGENS DO ALUNO
# =====================================================================
path_img1 = 'imagem1.png'
path_img2 = 'imagem2.png'

img1 = cv2.imread(path_img1)
img2 = cv2.imread(path_img2)

if img1 is None or img2 is None:
    raise FileNotFoundError("Verifique se as imagens proprias foram salvas corretamente no diretorio.")

# Conversao para escala de cinza para os extratores de feicoes
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# =====================================================================
# PASSO 1: DETECCAO DE CARACTERISTICAS E CORRESPONDENCIA (MATCHING)
# =====================================================================
# Utilizaremos o SIFT (Scale-Invariant Feature Transform)
sift = cv2.SIFT_create()

kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

# Forca bruta para encontrar os melhores matches usando a razao de Lowe
bf = cv2.BFMatcher()
matches_brutos = bf.knnMatch(des1, des2, k=2)

# Aplicacao do teste de razao de Lowe para filtrar matches muito ambiguos
bons_matches = []
for m, n in matches_brutos:
    if m.distance < 0.75 * n.distance:
        bons_matches.append(m)

# Extracao das coordenadas numericas dos pontos correspondentes
src_pts = np.float32([kp1[m.queryIdx].pt for m in bons_matches]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in bons_matches]).reshape(-1, 1, 2)

print(f"Total de pontos correspondentes validados pre-geometria: {len(bons_matches)}")

# =====================================================================
# PASSO 2: ALINHAMENTO 2D VIA MINIMOS QUADRADOS TRADICIONAIS (SEM RANSAC)
# =====================================================================
# O metodo classico tenta ajustar todos os pontos de uma vez (method=0)
H_ls, status_ls = cv2.findHomography(src_pts, dst_pts, method=0)

print("\n--- Matriz de Homografia via Minimos Quadrados Classicos ---")
print(H_ls)

# =====================================================================
# PASSO 3: MINIMOS QUADRADOS ROBUSTOS VIA RANSAC
# =====================================================================
# Definimos um limiar de reprojecao de 5 pixels para considerar um inlier
reproj_threshold = 5.0
H_ransac, mask_ransac = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, reproj_threshold)

print("\n--- Matriz de Homografia via RANSAC ---")
print(H_ransac)

# Contagem de Inliers e Outliers
inliers_count = np.sum(mask_ransac)
outliers_count = len(mask_ransac) - inliers_count
print(f"\nResultado RANSAC: {inliers_count} Inliers | {outliers_count} Outliers")

# Visualizacao das correspondencias aceitas pelo RANSAC
img_matches = cv2.drawMatches(img1, kp1, img2, kp2, bons_matches, None,
                              matchColor=(0, 255, 0), singlePointColor=None,
                              matchesMask=mask_ransac.ravel().tolist(), flags=2)

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
plt.title("Correspondencias (Inliers) Selecionadas pelo RANSAC")
plt.axis('off')
plt.show()

# =====================================================================
# PASSO 4: COSTURA DE IMAGENS (IMAGE STITCHING)
# =====================================================================
# Vamos rotacionar e projetar a Imagem 1 no plano da Imagem 2
width = img1.shape[1] + img2.shape[1]
height = max(img1.shape[0], img2.shape[0])

# Projecao usando Homografia por Minimos Quadrados
img1_warped_ls = cv2.warpPerspective(img1, H_ls, (width, height))

# Projecao usando Homografia pelo RANSAC
img1_warped_ransac = cv2.warpPerspective(img1, H_ransac, (width, height))

# Criacao do Mosaico (colando a imagem 2 por cima da zona de projecao)
mosaico_ls = img1_warped_ls.copy()
mosaico_ls[0:img2.shape[0], 0:img2.shape[1]] = img2

mosaico_ransac = img1_warped_ransac.copy()
mosaico_ransac[0:img2.shape[0], 0:img2.shape[1]] = img2

# Exibicao dos resultados comparativos
fig, ax = plt.subplots(2, 1, figsize=(14, 10))
ax[0].imshow(cv2.cvtColor(mosaico_ls, cv2.COLOR_BGR2RGB))
ax[0].set_title("Mosaico Final - Minimos Quadrados Tradicionais (Pode falhar se houver outliers)")
ax[0].axis('off')

ax[1].imshow(cv2.cvtColor(mosaico_ransac, cv2.COLOR_BGR2RGB))
ax[1].set_title("Mosaico Final - RANSAC Robusto")
ax[1].axis('off')

plt.tight_layout()
plt.show()

**Resultados obtidos pela equipe.**

A Figura 1 mostra as correspondências aprovadas pelo RANSAC (linhas verdes ligando os *inliers*) entre
o quadro da esquerda (uma pessoa da equipe) e o da direita (a mesma cena com um objeto texturizado
segurado na mão). Note que a maior parte dos inliers se concentra no **objeto texturizado** e na
**estrutura horizontal do ambiente** (a calha/parapeito metálico ao fundo), que são as regiões com
feições estáveis comuns às duas vistas.

![Correspondências (inliers) selecionadas pelo RANSAC](lab3_correspondencias_ransac.png)

*Figura 1 — Saída de `lab3_.py`: correspondências inliers aprovadas pelo RANSAC.*

A Figura 2 compara os dois mosaicos. No topo, o mosaico gerado com a homografia de **Mínimos Quadrados
Tradicionais**: a projeção da imagem 1 sai **distorcida/esticada** na zona à direita, efeito típico de
outliers contaminando o ajuste global. Embaixo, o mosaico com a homografia **RANSAC**: o alinhamento da
região sobreposta fica coerente, embora a área não coberta apareça em preto (sem *blending*).

![Comparação dos mosaicos: MQT (topo) e RANSAC (base)](lab3_mqt_ransac_robusto.png)

*Figura 2 — Saída de `lab3_.py`: mosaico por MQT (topo) versus RANSAC (base).*

### 4.3 (B) Pipeline de homografia e mosaico em duas webcams ao vivo

**Objetivo:** adaptar o programa do item (A) para ler **duas webcams pelo mesmo código** e executar o
pipeline (SIFT → BF *matching* → razão de Lowe → homografia RANSAC → *warp* → mosaico) **a cada par de
quadros**, mostrando o resultado em janelas de vídeo contínuas.

**Principais adaptações em relação ao item (A):**
- As entradas passam a ser `cv2.VideoCapture(0)` e `cv2.VideoCapture(1)`, processadas dentro de um laço
  `while`.
- **Robustez ao vivo:** se uma câmera não retorna descritores (`des1`/`des2` nulos) ou se há **menos de
  10** bons matches, o quadro é apenas exibido e o laço continua, evitando exceções quando a cena tem
  pouca textura.
- O resultado é exibido em duas janelas: **`Matches RANSAC`** (correspondências inliers entre os dois
  quadros) e **`Mosaico`** (panorama ao vivo), com a contagem de **inliers/outliers** sobreposta via
  `cv2.putText`. A tecla **`q`** encerra e libera as duas câmeras.

In [ ]:
import cv2
import numpy as np

# ==========================================================
# ABERTURA DAS DUAS WEBCAMS
# ==========================================================
cap1 = cv2.VideoCapture(0)
cap2 = cv2.VideoCapture(1)

if not cap1.isOpened():
    print("Erro ao abrir webcam 0")
    exit()

if not cap2.isOpened():
    print("Erro ao abrir webcam 1")
    exit()

sift = cv2.SIFT_create()
bf = cv2.BFMatcher()

while True:

    ret1, img1 = cap1.read()
    ret2, img2 = cap2.read()

    if not ret1 or not ret2:
        print("Erro na captura")
        break

    # CONVERSAO PARA CINZA
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    # DETECCAO DE FEATURES
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    if des1 is None or des2 is None:
        continue

    # MATCHING
    matches_brutos = bf.knnMatch(des1, des2, k=2)

    bons_matches = []
    for m, n in matches_brutos:
        if m.distance < 0.75 * n.distance:
            bons_matches.append(m)

    if len(bons_matches) < 10:
        cv2.imshow("Webcam 1", img1)
        cv2.imshow("Webcam 2", img2)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # PONTOS CORRESPONDENTES
    src_pts = np.float32([kp1[m.queryIdx].pt for m in bons_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in bons_matches]).reshape(-1, 1, 2)

    # HOMOGRAFIA VIA RANSAC
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if H is None:
        continue

    # VISUALIZACAO DOS MATCHES
    img_matches = cv2.drawMatches(img1, kp1, img2, kp2, bons_matches, None,
                                  matchColor=(0, 255, 0),
                                  matchesMask=mask.ravel().tolist(), flags=2)

    # CRIACAO DO MOSAICO
    largura = img1.shape[1] + img2.shape[1]
    altura = max(img1.shape[0], img2.shape[0])

    warped = cv2.warpPerspective(img1, H, (largura, altura))
    mosaico = warped.copy()
    mosaico[0:img2.shape[0], 0:img2.shape[1]] = img2

    # CONTAGEM DE INLIERS
    inliers = int(np.sum(mask))
    outliers = len(mask) - inliers

    cv2.putText(mosaico, f"Inliers: {inliers}  Outliers: {outliers}",
                (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # JANELAS
    cv2.imshow("Matches RANSAC", img_matches)
    cv2.imshow("Mosaico", mosaico)

    tecla = cv2.waitKey(1) & 0xFF
    if tecla == ord('q'):
        break

# FINALIZACAO
cap1.release()
cap2.release()
cv2.destroyAllWindows()

**Resultados obtidos pela equipe.**

A Figura 3 é uma captura da janela **`Matches RANSAC`** durante a execução ao vivo: os dois quadros das
webcams aparecem lado a lado, com as linhas verdes ligando os inliers — concentrados sobre o objeto
texturizado (livro/embalagem) comum às duas vistas.

![Janela ao vivo Matches RANSAC entre as duas webcams](Matches%20RANSAC_screenshot_17.06.2026.png)

*Figura 3 — Captura de `lab3_codigo_adaptado.py` (17/06/2026): correspondências RANSAC ao vivo.*

A Figura 4 mostra a janela **`Mosaico`** com a contagem sobreposta **`Inliers: 73  Outliers: 43`** —
números usados na análise estatística a seguir.

![Janela ao vivo Mosaico com contagem de inliers/outliers](lab3_mosaico.png)

*Figura 4 — Captura de `lab3_codigo_adaptado.py`: mosaico ao vivo com inliers/outliers.*

## 5. Análise e discussão

### 5.1 Análise estatística das características (outliers)

**Cálculo da proporção de outliers.** Tomando os números observados na execução ao vivo (Figura 4),
temos **73 inliers** e **43 outliers**, ou seja, um total de `bons_matches` = 73 + 43 = **116**
correspondências aprovadas pelo filtro de Lowe. Aplicando a fórmula do roteiro:

$$\%\text{Outliers} = \left(1 - \frac{\text{Inliers}}{\text{Total de Matches}}\right)\times 100 = \left(1 - \frac{73}{116}\right)\times 100 \approx 37{,}1\%$$

Portanto, cerca de **37%** das correspondências que haviam passado no teste de Lowe eram, na verdade,
**falsos positivos geométricos** descartados pelo RANSAC.

**Características visuais que geram falsos positivos.** Esse número relativamente alto se explica pelas
condições do ambiente de teste: (i) **texturas repetitivas** e superfícies lisas (paredes claras,
parapeito metálico, piso) produzem descritores SIFT muito parecidos entre si, fáceis de casar
erroneamente; (ii) **reflexos e brilhos** em superfícies metálicas e na tela/embalagem deslocam
keypoints entre as duas câmeras; (iii) **sombras** e variações de iluminação entre as duas webcams
alteram os gradientes locais; e (iv) elementos **não rígidos/dinâmicos** (a pessoa, a mão em movimento)
violam a hipótese de cena planar única, gerando correspondências geometricamente inconsistentes. O
RANSAC é exatamente o mecanismo que remove essas correspondências antes de estimar a homografia final.

### 5.2 Álgebra linear e homografia (comparação de matrizes)

**Por que $h_{22}$ é normalizado para 1,0.** A homografia atua em coordenadas **homogêneas** e está
definida **a menos de um fator de escala**: $H$ e $\lambda H$ produzem exatamente o mesmo mapeamento de
pontos. Assim, dos 9 elementos da matriz $3\times3$, apenas **8 são independentes** (8 graus de
liberdade). Para remover essa ambiguidade de escala fixa-se um elemento como referência — por convenção
$h_{22} = 1{,}0$ —, e os demais oito passam a ser expressos **relativamente** a ele. É por isso que,
tanto em `H_ls` quanto em `H_ransac`, o elemento inferior direito aparece (ou é normalizado para) 1,0.

**Diferença nos vetores de translação ($h_{02}$ e $h_{12}$).** A interpretação física é a parte central
desta questão: $h_{02}$ é a **translação horizontal** e $h_{12}$ a **translação vertical** estimadas
para alinhar a imagem 1 à imagem 2. Como o MQT eleva os erros ao quadrado, os **outliers** (≈37% das
correspondências, cf. 5.1) puxam o ajuste e **deslocam artificialmente** esses termos de translação —
é justamente o que se observa na Figura 2, em que o mosaico por MQT aparece esticado/deslocado na zona
projetada. O RANSAC, por usar só os inliers, devolve translações fisicamente coerentes com a real
sobreposição das vistas. A magnitude desse efeito é medida pelas diferenças absolutas:

$$\Delta h_{02} = \left| h_{02}^{\,LS} - h_{02}^{\,RANSAC} \right|, \qquad \Delta h_{12} = \left| h_{12}^{\,LS} - h_{12}^{\,RANSAC} \right|$$

> **Observação de reprodutibilidade.** Os valores numéricos exatos de $\Delta h_{02}$ e $\Delta h_{12}$
> devem ser lidos diretamente das matrizes `H_ls` e `H_ransac` **impressas no console** ao executar
> `lab3_.py` (o RANSAC é estocástico e a cena ao vivo varia, então os números mudam a cada execução).
> Qualitativamente, espera-se $\Delta h_{02}$ e $\Delta h_{12}$ **não desprezíveis** — da ordem de
> dezenas a centenas de pixels quando os outliers são significativos —, confirmando a distorção espacial
> introduzida pelo método clássico.

### 5.3 Análise geométrica e robótica

**Número mínimo de pares de pontos.** A matriz de homografia tem **8 graus de liberdade** (9 elementos
menos 1 da escala, com $h_{22}=1$). Como **cada correspondência** de ponto 2D fornece **duas equações**
lineares independentes (uma para $x'$ e outra para $y'$), são necessárias 8 equações e, portanto, no
mínimo $8/2 = \mathbf{4}$ **pares de pontos** correspondentes — desde que **não sejam colineares** (três
ou mais pontos sobre a mesma reta tornam o sistema degenerado). Com 4 pontos em posição geral o sistema
fica exatamente determinado; com mais pontos, sobredeterminado e resolvido por MQT/RANSAC.

**Superfície sem textura (robótica móvel).** Se um robô/drone aponta a câmera para uma **superfície
completamente uniforme** (uma parede branca, um terreno plano sem padrão), o SIFT **não encontra
keypoints distintos** — não há gradientes locais que individualizem pontos. Sem correspondências (ou com
pouquíssimas e ambíguas), `cv2.findHomography` **não consegue estimar** uma matriz válida (retorna
`None` ou uma homografia instável/degenerada). Na prática, a **navegação baseada em visão falha**: o
sistema perde a referência de movimento (odometria visual/SLAM sem *features* não consegue estimar
deslocamento), podendo travar ou derivar. Por isso, sistemas robóticos reais combinam a visão com
**outros sensores** (IMU, LIDAR, GPS) ou projetam **padrões/texturas artificiais** para garantir
feições rastreáveis.

### 5.4 Qualidade do processamento de imagem (artefatos da costura)

**Onde está a emenda.** No mosaico da Figura 2 (base, RANSAC) a junção ocorre na **fronteira vertical**
entre a `imagem2` (colada na posição original, à esquerda) e a `imagem1` projetada (*warped*, à
direita). Nessa linha de costura percebem-se os artefatos típicos: uma **linha de corte nítida**, um
**salto abrupto de iluminação** (as duas webcams têm balanços de branco/exposição distintos) e, na
região de sobreposição, leve **efeito fantasma** (*ghosting*) por pequenos erros residuais de
alinhamento e pela presença de elementos dinâmicos (a pessoa/mão).

**Técnica de pós-processamento (literatura de Szeliski).** O pipeline atual faz uma cópia "dura" de uma
imagem sobre a outra, sem suavização. A técnica clássica para minimizar essas discrepâncias é o
**blending multibanda** (*multi-band blending*), proposto por **Burt & Adelson (1983)** e tratado por
**Szeliski** em *Computer Vision: Algorithms and Applications*. A ideia é decompor cada imagem em uma
**pirâmide Laplaciana** (faixas de frequência) e fundir cada banda com uma **máscara de transição**
de largura adequada à frequência: as **baixas frequências** (iluminação global) são misturadas de forma
**suave e larga**, eliminando o salto de brilho, enquanto as **altas frequências** (bordas, detalhes)
são combinadas em uma faixa **estreita**, preservando a nitidez e evitando *ghosting*. Alternativas
mencionadas na literatura incluem o **feathering / alpha blending** (transição linear de pesos na
emenda, mais simples) e o **Poisson / gradient-domain blending** (costura no domínio do gradiente), que
também atenua as descontinuidades de iluminação na junção.

### 5.5 Aplicações e uso no trabalho final

Alinhamento, homografia e costura sustentam aplicações amplamente documentadas: **panoramas e
fotografia computacional**; **mapeamento aéreo e sensoriamento remoto** (mosaicos de imagens de drone
de terrenos planos); **realidade aumentada** (registro de conteúdo virtual sobre planos reais);
**odometria visual e SLAM** em robótica móvel e veículos autônomos; **estabilização de vídeo**; e
**registro de imagens médicas/de satélite**. O fio condutor é sempre o mesmo: estimar de forma robusta
a transformação geométrica entre vistas, lidando com ruído e outliers.

Para o **trabalho final**, a equipe pretende reutilizar este pipeline — SIFT + *matching* + razão de
Lowe + **homografia robusta via RANSAC** + *warping* — como base para **alinhar/registrar vistas** de
uma cena capturada por câmera e construir um mosaico ou sobrepor informação. A principal lição
incorporada ao projeto é a **necessidade do RANSAC** (e não do MQT puro) em qualquer cenário real e
ruidoso, além da etapa de **blending** (multibanda) para entregar um mosaico visualmente limpo. Caso o
desempenho em tempo real seja crítico, consideraremos detectores mais leves (ORB) como no Lab 2.

## 6. Conclusões

O Laboratório 3 cumpriu seus objetivos. A equipe compreendeu e aplicou a **homografia 2D** para alinhar
imagens, implementou o pipeline completo de **Image Stitching** e, sobretudo, comparou na prática o
**Mínimos Quadrados Tradicionais** com o **RANSAC**. Ficou evidente que, em um cenário real e ruidoso
(≈37% de outliers nos nossos testes), o MQT **distorce** a homografia — especialmente os termos de
translação $h_{02}$ e $h_{12}$ — produzindo mosaicos deformados, enquanto o RANSAC, ao ajustar apenas
os **inliers**, devolve um alinhamento geometricamente coerente. Confirmamos também que são necessários
**4 pares de pontos** não colineares para estimar a homografia (8 graus de liberdade, 2 equações por
ponto), que superfícies **sem textura** inviabilizam a estimativa e comprometem a navegação visual, e
que a emenda do mosaico exige **blending** (idealmente multibanda) para eliminar linha de corte, saltos
de iluminação e *ghosting*. O conjunto forma uma base robusta e diretamente reaproveitável no trabalho
final.

## 7. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026** (diretrizes de integridade na pesquisa quanto ao uso de
Inteligência Artificial Generativa — IAG), a equipe declara, de forma transparente, o seguinte:

- **Ferramenta utilizada:** assistente de IA generativa (modelo de linguagem da Anthropic — Claude).
- **Finalidade:** apoio à **redação e organização textual** deste relatório (estruturação das seções,
  revisão de texto e formatação do notebook) e à **explicação didática** dos conceitos teóricos
  (homografia, RANSAC, *blending*).
- **Autoria e responsabilidade:** o **planejamento, a implementação dos códigos OpenCV, a captura das
  imagens e a execução dos experimentos** foram realizados pelos integrantes da equipe. O conteúdo
  técnico foi **revisado e validado** pelos autores, que se **responsabilizam integralmente** pelo
  resultado final, conforme as alíneas (d) e (f) da referida portaria. Nenhum conteúdo gerado por IAG
  foi submetido como se fosse de autoria exclusivamente humana sem esta devida declaração.

## 8. Referências

1. OpenCV — *Feature Matching + Homography to find Objects*. Disponível em:
   https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html
2. OpenCV — *Basic concepts of the homography explained with code*. Disponível em:
   https://docs.opencv.org/4.x/d9/dab/tutorial_homography.html
3. MALLICK, S. *Homography Examples using OpenCV (Python / C++)*. LearnOpenCV. Disponível em:
   https://learnopencv.com/homography-examples-using-opencv-python-c/
4. GeeksforGeeks — *Image Stitching with OpenCV*. Disponível em:
   https://www.geeksforgeeks.org/computer-vision/image-stitching-with-opencv/
5. ROSEBROCK, A. *OpenCV panorama stitching*. PyImageSearch, 2016. Disponível em:
   https://pyimagesearch.com/2016/01/11/opencv-panorama-stitching/
6. SZELISKI, R. *Computer Vision: Algorithms and Applications*. 2ª ed. Springer, 2022. (Cap. de
   *Image alignment and stitching*; *multi-band blending*).
7. FISCHLER, M. A.; BOLLES, R. C. *Random Sample Consensus (RANSAC): A Paradigm for Model Fitting...*.
   Communications of the ACM, v. 24, n. 6, p. 381–395, 1981.
8. BURT, P. J.; ADELSON, E. H. *A Multiresolution Spline With Application to Image Mosaics*. ACM
   Transactions on Graphics, v. 2, n. 4, p. 217–236, 1983.
9. LOWE, D. G. *Distinctive Image Features from Scale-Invariant Keypoints*. IJCV, v. 60, n. 2,
   p. 91–110, 2004.
10. CNPq — *Portaria nº 2664/2026* (integridade na pesquisa e uso de IA generativa). Disponível em:
    http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775